In [1]:
import os
from pathlib import Path

PROJECT = Path(r"C:\2nd Disk\intern\price-comparision")

os.environ["JAVA_HOME"] = str(
    PROJECT / "OpenJDK17U-jdk_x64_windows_hotspot_17.0.19_10" / "jdk-17.0.19+10"
)
os.environ["HADOOP_HOME"] = str(PROJECT / "hadoop")
os.environ["SPARK_HOME"] = str(PROJECT / ".venv" / "Lib" / "site-packages" / "pyspark")
os.environ["PYSPARK_PYTHON"] = str(PROJECT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT / ".venv" / "Scripts" / "python.exe")

os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + r"\bin;"
    + os.environ["HADOOP_HOME"] + r"\bin;"
    + os.environ["PATH"]
)

print(os.environ["JAVA_HOME"])
print(os.environ["HADOOP_HOME"])

C:\2nd Disk\intern\price-comparision\OpenJDK17U-jdk_x64_windows_hotspot_17.0.19_10\jdk-17.0.19+10
C:\2nd Disk\intern\price-comparision\hadoop


# Đọc Hudi từ MinIO

Notebook chỉ đọc dữ liệu từ máy 1.

In [2]:
from pyspark.sql import SparkSession

# Nếu đã tạo SparkSession trước đó thì dừng nó
try:
    spark.stop()
except:
    pass

spark = (
    SparkSession.builder
    .appName("read_hudi_from_minio")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.blockManager.port", "0")
    .config(
        "spark.jars.packages",
        "org.apache.hudi:hudi-spark3.5-bundle_2.12:1.2.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4",
    )
    .config(
        "spark.jars.ivy",
        r"C:\2nd Disk\intern\price-comparision\.ivy2",
    )
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["MINIO_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_SECRET_KEY"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem",
    )
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.blockManager.port", "0")
    .getOrCreate()
)

print("Spark:", spark.version)

Spark: 3.5.6


In [3]:
# Discover all Gold Hudi tables currently published by DE.
# Some fact/dimension tables are retailer-scoped (for example, .../store=go),
# so this does not assume one fixed table path.
import boto3
from botocore.config import Config

BUCKET = "supermarket-lakehouse"
GOLD_PREFIX = "gold/"
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    config=Config(signature_version="s3v4", s3={"addressing_style": "path"}),
)

# A published Gold Hudi root is either gold/<table>_hudi or
# gold/<table>_hudi/store=<retailer>. This excludes legacy run exports
# and Hudi's internal .hoodie/metadata tables.
hudi_roots = set()
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=GOLD_PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        marker = "/.hoodie/hoodie.properties"
        if key.endswith(marker):
            root = key[: -len(marker)]
            parts = root.removeprefix(GOLD_PREFIX).split("/")
            is_global_table = len(parts) == 1 and parts[0].endswith("_hudi")
            is_store_table = (
                len(parts) == 2
                and parts[0].endswith("_hudi")
                and parts[1].startswith("store=")
            )
            if is_global_table or is_store_table:
                hudi_roots.add(root)

if not hudi_roots:
    raise RuntimeError(f"No Hudi table found under s3a://{BUCKET}/{GOLD_PREFIX}")

tables = {}
summary = []
failed_tables = []
for root in sorted(hudi_roots):
    name = root.removeprefix(GOLD_PREFIX)
    path = f"s3a://{BUCKET}/{root}"
    try:
        df = spark.read.format("hudi").load(path)
        tables[name] = df
        summary.append((name, df.count(), len(df.columns), path))
    except Exception as exc:
        failed_tables.append((name, type(exc).__name__, str(exc)))

summary_df = spark.createDataFrame(
    summary,
    ["table", "rows", "column_count", "path"],
).orderBy("table")
summary_df.show(truncate=False)
print(f"Loaded {len(tables)} Hudi tables / {sum(row[1] for row in summary):,} rows")
if failed_tables:
    print("Tables that could not be read:")
    for name, error_type, message in failed_tables:
        print(f"- {name}: {error_type} — {message}")

# Inspect a table, e.g. tables["fact_price_snapshot_daily_hudi/store=bachhoaxanh"].show(20, truncate=False)

+------------------------------------------------+----+------------+---------------------------------------------------------------------------------+
|table                                           |rows|column_count|path                                                                             |
+------------------------------------------------+----+------------+---------------------------------------------------------------------------------+
|dim_date_hudi                                   |1   |13          |s3a://supermarket-lakehouse/gold/dim_date_hudi                                   |
|dim_product_hudi                                |153 |17          |s3a://supermarket-lakehouse/gold/dim_product_hudi                                |
|dim_promotion_hudi/store=bachhoaxanh            |1400|19          |s3a://supermarket-lakehouse/gold/dim_promotion_hudi/store=bachhoaxanh            |
|dim_promotion_hudi/store=go                     |388 |19          |s3a://supermarket-lakehous